# Lesson 7: Observability & Debugging

## What You'll Learn

1. **Why observability matters** — Agents are opaque by default
2. **Structured tracing** — Track every step of an agent run
3. **Cost tracking** — Know exactly how much each query costs
4. **Performance analysis** — P50/P95 latency, token usage trends
5. **Error analysis** — Categorize and debug failures
6. **PydanticAI message inspection** — Reading raw tool call traces
7. **Production integrations** — Logfire and Langfuse overview

---

## The Observability Problem

Agents are **non-deterministic black boxes**. Without tracing:
- You can't tell WHY an answer is wrong (bad SQL? wrong tool? hallucination?)
- You can't optimize cost (which queries burn the most tokens?)
- You can't debug intermittent failures
- You have no performance baselines

### Three Pillars of Agent Observability

| Pillar | What | Tools |
|--------|------|-------|
| **Traces** | Step-by-step execution record | Custom tracer, Logfire, Langfuse |
| **Metrics** | Aggregated numbers (latency, cost, success rate) | Dashboards, alerting |
| **Logs** | Detailed debug output | Structured logging |

---

## Setup

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import time
import json
import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.observability.tracing import Span, Trace, AgentTracer

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

---

## Part 1: Manual Tracing — Understanding the Building Blocks

Before using any framework, let's build tracing from scratch to understand
exactly what happens during an agent run.

In [ ]:
# -----------------------------------------------------------------
# Spans and Traces — the core primitives
# -----------------------------------------------------------------

# A Span represents one unit of work
span = Span(name="run_sql", span_type="tool_call", input_data="SELECT SUM(revenue) FROM sales")
time.sleep(0.1)  # Simulate work
span.finish(output="45230.50")

print("Single span:")
print(json.dumps(span.to_dict(), indent=2))

# A Trace groups related spans
trace = Trace(trace_id="demo_001", question="What is the total revenue?", model="gpt-4o-mini")

# Span 1: LLM decides to call a tool
s1 = Span(name="llm_decision", span_type="llm_call", input_data="What is the total revenue?")
time.sleep(0.05)
s1.finish(output="I should run a SQL query to sum the revenue column.")
s1.tokens_used = 150
trace.add_span(s1)

# Span 2: Tool execution
s2 = Span(name="run_sql", span_type="tool_call", input_data="SELECT SUM(revenue) FROM sales")
time.sleep(0.02)
s2.finish(output="45230.50")
trace.add_span(s2)

# Span 3: LLM synthesizes final answer
s3 = Span(name="synthesize", span_type="llm_call", input_data="SQL result: 45230.50")
time.sleep(0.03)
s3.finish(output="The total revenue is $45,230.50.")
s3.tokens_used = 100
trace.add_span(s3)

trace.finish(answer="The total revenue is $45,230.50.")

print(f"\n{trace.summary()}")
print(f"\nTimeline:")
for s in trace.spans:
    print(f"  [{s.span_type:>10}] {s.name:<20} {s.duration_ms:>6.1f}ms  tokens={s.tokens_used}")

---

## Part 2: Traced Agent — Automatic Instrumentation

Now let's wire the tracer into a real agent so every tool call and
LLM interaction is captured automatically.

In [ ]:
# -----------------------------------------------------------------
# Agent with built-in tracing
# -----------------------------------------------------------------

@dataclass
class TracedDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    tracer: AgentTracer = field(default_factory=AgentTracer)
    current_trace: Trace | None = None


traced_agent = Agent(
    "openai:local-model",
    deps_type=TracedDeps,
    system_prompt=(
        "You are a data analyst. Answer questions using SQL.\n"
        "Always include specific numbers. Be concise."
    ),
    retries=2,
)


@traced_agent.system_prompt
def add_context(ctx: RunContext[TracedDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@traced_agent.tool
def run_sql(ctx: RunContext[TracedDeps], query: str) -> str:
    """Run SQL against available tables using DuckDB."""
    # Start a span for this tool call
    trace = ctx.deps.current_trace
    span = None
    if trace:
        span = ctx.deps.tracer.start_span(trace, "run_sql", "tool_call", query)
    
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(query).fetchdf()
        output = result.to_string()
        if span:
            span.finish(output=output)
        return output
    except Exception as e:
        if span:
            span.finish(error=str(e))
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


def traced_query(deps: TracedDeps, question: str) -> str:
    """Run a query with full tracing."""
    # Start trace
    trace = deps.tracer.start_trace(question, model="openai:local-model")
    deps.current_trace = trace
    
    try:
        result = traced_agent.run_sync(question, deps=deps)
        
        # Estimate tokens from messages
        for msg in result.all_messages():
            msg_str = str(msg)
            token_est = len(msg_str) // 4
            trace.total_tokens += token_est
        
        # Estimate cost
        deps.tracer.estimate_cost(trace)
        trace.finish(answer=result.output, success=True)
        return result.output
    except Exception as e:
        trace.finish(answer="", success=False)
        return f"Error: {e}"


print("Traced agent ready.")

In [ ]:
# -----------------------------------------------------------------
# Run several queries with tracing
# -----------------------------------------------------------------

deps = TracedDeps(
    tables={"sales": sales_df, "employees": employees_df},
)

questions = [
    "What is the total revenue in the sales data?",
    "Which region has the highest revenue?",
    "What is the average salary by department?",
    "How many products have revenue above 1000?",
    "What is the correlation between quantity and revenue?",
]

for q in questions:
    print(f"\nQ: {q}")
    answer = traced_query(deps, q)
    print(f"A: {answer[:120]}")

print(f"\nTotal traces recorded: {len(deps.tracer.traces)}")

---

## Part 3: The Performance Dashboard

With traces collected, we can now build a comprehensive view of agent performance.

In [ ]:
# -----------------------------------------------------------------
# Dashboard: aggregate view of all runs
# -----------------------------------------------------------------

print(deps.tracer.dashboard())

In [ ]:
# -----------------------------------------------------------------
# Detailed per-trace analysis
# -----------------------------------------------------------------

print("Per-trace breakdown:\n")
print(f"{'Question':<50} {'Duration':>10} {'Tokens':>8} {'Tools':>6} {'Cost':>10}")
print("-" * 88)

for trace in deps.tracer.traces:
    print(
        f"{trace.question[:48]:<50} "
        f"{trace.duration_ms:>8.0f}ms "
        f"{trace.total_tokens:>8} "
        f"{len(trace.tool_calls):>6} "
        f"${trace.total_cost_usd:>8.4f}"
    )

In [ ]:
# -----------------------------------------------------------------
# Inspect the slowest trace in detail
# -----------------------------------------------------------------

slowest = deps.tracer.get_slowest(1)
if slowest:
    t = slowest[0]
    print(f"SLOWEST TRACE")
    print(f"{'='*50}")
    print(t.summary())
    print(f"\nSpan timeline:")
    for s in t.spans:
        status = "ERR" if s.error else "OK"
        print(f"  [{s.span_type:>10}] {s.name:<20} {s.duration_ms:>7.1f}ms [{status}]")
        if s.input_data:
            print(f"              Input:  {s.input_data[:80]}")
        if s.output_data:
            print(f"              Output: {s.output_data[:80]}")

---

## Part 4: PydanticAI Message Inspection

PydanticAI stores the full message history of each run. This is the
most detailed trace you can get — every LLM prompt and response,
every tool call and return.

In [ ]:
# -----------------------------------------------------------------
# PydanticAI native message inspection
# -----------------------------------------------------------------

simple_agent = Agent(
    "openai:local-model",
    deps_type=TracedDeps,
    system_prompt="You are a data analyst. Use SQL to answer questions. Be concise.",
    retries=2,
)


@simple_agent.system_prompt
def ctx(ctx: RunContext[TracedDeps]) -> str:
    lines = ["\nTables:"]
    for name, df in ctx.deps.tables.items():
        lines.append(f"  '{name}': {len(df)} rows, cols: {list(df.columns)}")
    return "\n".join(lines)


@simple_agent.tool
def query_sql(ctx: RunContext[TracedDeps], sql: str) -> str:
    """Run SQL query against tables."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        return conn.execute(sql).fetchdf().to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


# Run a query and inspect the messages
deps2 = TracedDeps(tables={"sales": sales_df, "employees": employees_df})
result = simple_agent.run_sync("Which product has the highest total revenue?", deps=deps2)

print(f"Answer: {result.output}\n")
print(f"Total messages in history: {len(result.all_messages())}\n")

# Inspect each message
for i, msg in enumerate(result.all_messages()):
    msg_type = type(msg).__name__
    msg_str = str(msg)[:150]
    print(f"  [{i}] {msg_type}")
    print(f"      {msg_str}")
    print()

In [ ]:
# -----------------------------------------------------------------
# Parse message types to understand the agent's reasoning flow
# -----------------------------------------------------------------

print("Agent reasoning flow:\n")
step = 0
for msg in result.all_messages():
    msg_type = type(msg).__name__
    
    if "Request" in msg_type and "System" not in msg_type:
        # User or system prompt
        step += 1
        content = str(msg)
        if "UserPrompt" in msg_type:
            print(f"  Step {step}: USER PROMPT")
            # Extract the user's question from the message
            print(f"          -> (user question sent to LLM)")
    
    elif "Response" in msg_type or "Model" in msg_type:
        step += 1
        msg_str = str(msg)
        
        # Check if this contains tool calls
        if "ToolCall" in msg_str or "tool_call" in msg_str or "tool-call" in msg_str:
            print(f"  Step {step}: LLM DECIDES to call tool")
            # Try to extract tool name
            if "query_sql" in msg_str or "run_sql" in msg_str:
                print(f"          -> Calling SQL tool")
        elif "ToolReturn" in msg_str or "tool_return" in msg_str or "tool-return" in msg_str:
            print(f"  Step {step}: TOOL RESULT returned")
            print(f"          -> {msg_str[:100]}")
        else:
            print(f"  Step {step}: LLM RESPONSE")
            print(f"          -> {msg_str[:100]}")

print(f"\nTotal steps: {step}")

---

## Part 5: Cost Tracking in Detail

In [ ]:
# -----------------------------------------------------------------
# Token estimation and cost calculation
# -----------------------------------------------------------------

from analyst.observability.tracing import MODEL_PRICING

print("Model pricing (per 1M tokens):")
print(f"{'Model':<30} {'Input':>10} {'Output':>10}")
print("-" * 52)
for model, (inp, out) in MODEL_PRICING.items():
    print(f"{model:<30} ${inp:>8.2f} ${out:>8.2f}")

# Cost breakdown for our traced queries
print(f"\n\nCost breakdown for {len(deps.tracer.traces)} traced queries:")
total_cost = sum(t.total_cost_usd for t in deps.tracer.traces)
total_tokens = sum(t.total_tokens for t in deps.tracer.traces)

print(f"  Total tokens:  {total_tokens:,}")
print(f"  Total cost:    ${total_cost:.4f}")
print(f"  Cost per query: ${total_cost / len(deps.tracer.traces):.4f}")

# Project monthly cost
queries_per_day = 100
monthly_cost = total_cost / len(deps.tracer.traces) * queries_per_day * 30
print(f"\n  Projected monthly cost at {queries_per_day} queries/day: ${monthly_cost:.2f}")

---

## Part 6: Error Analysis & Debugging

In [ ]:
# -----------------------------------------------------------------
# Simulate some failures to see error analysis
# -----------------------------------------------------------------

# Query that will likely cause issues (ambiguous table reference)
tricky_questions = [
    "What is the average of column xyz?",  # Non-existent column
    "Join sales and employees on employee_id",  # No common key
]

for q in tricky_questions:
    print(f"\nQ: {q}")
    answer = traced_query(deps, q)
    print(f"A: {answer[:150]}")

# Error summary
errors = deps.tracer.get_error_summary()
if errors:
    print(f"\nError categories:")
    for err_type, count in errors.items():
        print(f"  {err_type}: {count}")

failures = deps.tracer.get_failures()
print(f"\nFailed traces: {len(failures)}/{len(deps.tracer.traces)}")
for f in failures:
    print(f"  - {f.question[:60]}")
    for s in f.errors:
        print(f"    Error in {s.name}: {s.error[:80]}")

---

## Part 7: Saving & Analyzing Traces

In [ ]:
# -----------------------------------------------------------------
# Save traces for post-hoc analysis
# -----------------------------------------------------------------

traces_path = Path("../data/traces.json")
deps.tracer.save(traces_path)
print(f"Saved {len(deps.tracer.traces)} traces to {traces_path}")

# Load and analyze
saved_traces = json.loads(traces_path.read_text())
print(f"\nSample trace:")
print(json.dumps(saved_traces[0], indent=2)[:500])

In [ ]:
# -----------------------------------------------------------------
# Analyze traces as a DataFrame (production pattern)
# -----------------------------------------------------------------

trace_data = []
for t in saved_traces:
    trace_data.append({
        "trace_id": t["trace_id"],
        "question": t["question"][:50],
        "duration_ms": t["duration_ms"],
        "tokens": t["total_tokens"],
        "cost_usd": t["total_cost_usd"],
        "success": t["success"],
        "n_spans": len(t["spans"]),
    })

traces_df = pd.DataFrame(trace_data)
print("Traces as DataFrame:\n")
print(traces_df.to_string(index=False))

print(f"\nSummary statistics:")
print(traces_df[["duration_ms", "tokens", "cost_usd"]].describe().round(2))

---

## Part 8: Production Integration — Logfire & Langfuse

Our custom tracer is great for learning, but production systems use
dedicated observability platforms.

### Logfire (PydanticAI native)

Logfire is built by the Pydantic team and integrates with PydanticAI with
zero configuration:

```python
import logfire

logfire.configure()  # Uses LOGFIRE_TOKEN from .env
logfire.instrument_pydantic_ai()  # Auto-trace all PydanticAI agents

# That's it! Every agent.run_sync() is now traced in Logfire dashboard
agent = Agent("openai:local-model", system_prompt="...")
result = agent.run_sync("What is the total revenue?")  # Auto-traced
```

Logfire provides:
- Automatic span creation for LLM calls and tool calls
- Token counting and cost estimation
- Visual trace waterfall diagrams
- Performance dashboards

### Langfuse (Open-source)

Langfuse is an open-source LLM observability platform with broader
ecosystem support:

```python
from langfuse import Langfuse

langfuse = Langfuse()  # Uses LANGFUSE_* env vars

# Manual tracing with Langfuse
trace = langfuse.trace(name="data_analysis")
span = trace.span(name="llm_call", input={"question": "..."})
# ... run agent ...
span.end(output={"answer": "..."})
trace.update(output={"final_answer": "..."})
```

Langfuse provides:
- Trace visualization and search
- Eval dataset management
- A/B testing across model versions
- Cost analytics and alerting

### Which to Use?

| Feature | Logfire | Langfuse |
|---------|---------|----------|
| PydanticAI integration | Zero-config | Manual |
| Open source | No (hosted) | Yes (self-host or cloud) |
| Eval features | Basic | Advanced |
| Cost | Free tier available | Free self-hosted |
| Best for | PydanticAI-native stacks | Multi-framework stacks |

---

## Summary

| Concept | Implementation | Why |
|---------|---------------|-----|
| Spans | `Span` class with timing/metadata | Track individual operations |
| Traces | `Trace` groups spans per query | End-to-end view of agent run |
| AgentTracer | Collects traces, computes stats | Aggregate performance view |
| Message inspection | `result.all_messages()` | Debug exact LLM interactions |
| Cost tracking | Token count * model pricing | Budget control |
| Error analysis | `get_failures()`, `get_error_summary()` | Debug and improve |
| Persistence | Save traces to JSON | Post-hoc analysis |

### Production Patterns

1. **Trace everything** — You can't debug what you can't see
2. **Track latency percentiles** — P50 for typical, P95/P99 for worst case
3. **Set cost budgets** — Alert when per-query cost exceeds threshold
4. **Categorize errors** — Know which failure modes are most common
5. **Save traces** — You'll want them for debugging issues reported later
6. **Dashboard daily** — Catch regressions early (latency creep, cost spikes)

**Next: Lesson 8 — Full Agent Assembly (putting everything together)**

---

## Exercises

1. **Logfire integration** — Install logfire and instrument the traced agent. Compare the dashboard to our custom one.
2. **Alerting** — Add a function that raises a warning when P95 latency exceeds a threshold.
3. **Cost budget enforcement** — Modify the agent to stop mid-run if token cost exceeds a limit.
4. **Trace comparison** — Build a function that compares two sets of traces (before/after a change) and highlights regressions.
5. **Visual timeline** — Use matplotlib to create a Gantt-chart-style visualization of span timings.